<a href="https://colab.research.google.com/github/elianpuuig/Ciencia-De-Datos/blob/main/Aluar_PP1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#yfinancepara descargar los datos reales de Aluar desde Yahoo Finance
!pip install yfinance

#ydata-profiling para reporte automatico inteligente
!pip install ydata-profiling

Obtuve los datos desde yfinance , son datasets dinamicos ya que se conectan directamente a la API publica de Yahoo Finance , "BA" significa : (Bolsas y Mercados Argentinos)

yfinance mide el valor financiero diario que los inversores globales y locales le otorgan a la empresa minuto a minuto en la bolsa.

In [ ]:
import pandas as pd
import yfinance as yf
from ydata_profiling import ProfileReport
from IPython.display import HTML

#datos de aluar desde el 2020 en adelante
data = yf.download('ALUA.BA',start="2020-01-01")
df_aluar = pd.DataFrame(data)
df_aluar

# Análisis Exploratorio Automático (EDA) con "ydata-profiling"

*En este reporte podemos ver como el volumen ( Datos de la columna "volume" que representa la cantidad de acciones que se compro y vendio en la bolsa  la empresa ese mismo dia ) no sube ni baja en correlacion al precio de las acciones, esto es por que justamente si un dia hay un volumen enorme de operaciones, puede ser porque todos quieren comprar (haciendo subir el precio) o porque hay panico y todos quieren vender (haciendo caer fuertemente el precio). Por eso el volumen no tiene una correlacion directa con que el precio sea siempre alto.*

In [ ]:
reporte = ProfileReport(df_aluar, title="Reporte De Datos Automatico: Aluar")

reporte.to_file('reporte_aluar_automatico.html')

with open("reporte_aluar_automatico.html","r", encoding="utf-8") as f:
    html_content = f.read()

HTML(html_content)


#Análisis de Rendimiento Dinámico (Plotly express)

Se calculo el cambio diario que basicamente es ver si la accion subio o bajo respecto al dia anterior, como el mercado es variable calcule primero

*la mediana movil de 20 dias para suavizar la curva en el grafico y ver mas que nada la tendencia de las acciones a final de mes*

*y tambien Calcule la banda de volatilidad : que seria la linea limite que la accion deberia moverse en esos 20 dias*

**Grafico hecho con (Plotly Express)**


Sirve para medir la "temperatura" del mercado. Si la línea del rendimiento diario va por debajo del límite de volatilidad, la acción está "sana" y tranquila. Pero si el rendimiento perfora ese techo técnico, se te enciende una alarma: el mercado se volvió loco ese día.

In [ ]:
import plotly.express as px



df_aluar = data.copy()
# Aca avalue si es una columna multi index osea anidada
# en ese caso me quedo solo con el primer nivel financiero
if isinstance(df_aluar.columns, pd.MultiIndex):
    df_aluar.columns = df_aluar.columns.get_level_values(0)

#cambio diario sobre el precio de cierre
df_aluar["Cambio_Diario"] = df_aluar["Close"].pct_change()

#Mediana Movil de la columna CAMBIO DIARIO
# Queremos ver la tendencia del rendimiento, por eso usamos 'Cambio_Diario' aquí:
df_aluar['Median_20'] = df_aluar["Cambio_Diario"].rolling(window=20).median()

#Calcule la volatilidad (Desviación Estándar movil)
df_aluar['Volatilidad_Alta'] = df_aluar['Median_20'] + df_aluar['Cambio_Diario'].rolling(
    window=20).std()

# Elimine nulos
dfAluar_clean = df_aluar.dropna(subset=['Cambio_Diario', 'Median_20', 'Volatilidad_Alta'])

fig_rendimiento = px.line(
    dfAluar_clean,
    x=dfAluar_clean.index,
    y=['Cambio_Diario', 'Median_20', 'Volatilidad_Alta'],
    labels={
        'value': 'Rendimiento',
        'variable': 'Métricas 📊',
        'Date': 'Fecha'
    },
    title="Análisis Avanzado de Rendimiento y Volatilidad: Aluar",
    template="plotly_dark",
    color_discrete_sequence=["#00ffff", "#ff00ff", "#ffff00"]
)

fig_rendimiento.update_layout(
    yaxis=dict(tickformat='.2%'),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    hovermode="x unified"
)

fig_rendimiento.add_hline(y=0, line_dash="longdashdot", line_width=2, line_color="blue")

fig_rendimiento.show()

#Distribución de Volatilidad Anual (Altair)
Estadística Avanzada

se hizo un Boxplot (Gráfico de Caja).
La "caja" encierra el comportamiento común de la acción (el 50% de los días de ese año). Los puntos sueltos que quedan muy arriba o muy abajo del gráfico son los outliers (los días totalmente anormales, en los que pudo haber subas en especificamente altas).

 Boxplot es una herramnienta que nos sirve para comparar muchos años de forma rapida

las cajas cortas y aplastadas nos indican que aluar fue en ese año/años bastante estable

todo lo contrario a una caja alargada con puntos flotando en el aire (anomalias) significa que ese año hubo muchas subas y bajas en cuanto a acciones

In [ ]:
import altair as alt

# Preparamos los datos para Altair
df_box = dfAluar_clean['Cambio_Diario'].reset_index()
df_box.columns = ['Fecha', 'Variacion']

# obtengo cada año para crear multiples boxplots por año
df_box['Año'] = df_box['Fecha'].dt.year


aluar_variacion = alt.Chart(df_box).mark_boxplot(
    color="#00ffff",
    size=40,
    outliers=alt.MarkConfig(color="#ff00ff")
).encode(
    x=alt.X('Año:O', title='Año', axis=alt.Axis(labelAngle=0, grid=False)),
    y=alt.Y('Variacion:Q', axis=alt.Axis(format='%', title='Cambio Diario', grid=False))
).properties(
    title="Volatilidad Anual de Aluar",
    width=700,
    height=500
).configure(
    background='black',
).configure_axis(
    labelColor='white',
    titleColor='white',
    domainColor='white',
    tickColor='white',
    grid=False
).configure_title(
    color='white',
    fontSize=18
)

aluar_variacion.show()

# Simulador de Retorno Acumulado (bokeh)
**Grafico con bokeh**

Basicamente vemos cuan rentable fue invertir en la empresa.

Para eso se Calculo el interes compuesto (multiplicando los rendimientos dia a dia)

Este analisis muestra el impacto real de la inflación y las ganancias en el tiempo, permitiéndole a una persona común jugar con su propio presupuesto y entender la rentabilidad de Aluar de forma 100% interactiva.

In [ ]:
from bokeh.plotting import figure, show
from bokeh.models import HoverTool, NumeralTickFormatter, Slider, CustomJS, ColumnDataSource
from bokeh.layouts import column
from bokeh.io import output_notebook

output_notebook()

df_SRA = dfAluar_clean.copy()


#rendimiento, el factor acumulado y limpiamos nulos

df_SRA['Factor_Acumulado'] = (1 + df_SRA['Cambio_Diario']).cumprod()

#inversión inicial
inversion_inicial = 10000
df_SRA['Capital_Evolucion'] = df_SRA['Factor_Acumulado'] * inversion_inicial
df_SRA['Fecha_Texto'] = df_SRA.index.strftime('%Y-%m-%d')

#datos para el grafico con bokeh
source = ColumnDataSource(df_SRA)

p = figure(
    title="💰 SIMULADOR INTERACTIVO DE INVERSIÓN: ALUAR",
    x_axis_type='datetime',
    width=750,
    height=450,
    tools="pan,wheel_zoom,box_zoom,reset,save",
    background_fill_color="black",
    border_fill_color="#0a0a0a"
)

# La curva interactiva
curva_capital = p.line(
    x='Date',
    y='Capital_Evolucion',
    source=source,
    line_width=2.5,
    line_color="#feaf00",
    legend_label="Capital en Pesos ($)"
)

#cartel Hover
hover = HoverTool(
    renderers=[curva_capital],
    tooltips=[
        ("Fecha 📅", "@Fecha_Texto"),
        ("Capital Actual 💵", "$@Capital_Evolucion{0,0.00}")
    ],
    mode='vline'
)
p.add_tools(hover)

p.title.text_color = "white"
p.title.text_font_size = "13pt"

p.xaxis.axis_label = "Línea de Tiempo"
p.xaxis.axis_label_text_color = "white"
p.xaxis.major_label_text_color = "white"

p.yaxis.axis_label = "Valor del Portafolio (ARS)"
p.yaxis.axis_label_text_color = "white"
p.yaxis.major_label_text_color = "white"
p.yaxis.formatter = NumeralTickFormatter(format="$0,0")

p.xgrid.grid_line_color = None
p.ygrid.grid_line_color = None

p.legend.background_fill_color = "black"
p.legend.label_text_color = "white"
p.legend.location = "top_left"

# Slider interactivo 🎛️
slider = Slider(
    start=1000,
    end=100000,
    value=inversion_inicial,
    step=1000,
    title="Monto a Invertir En (ARS) 💵")

#logica js para que actualice en vivo el slider
callback = CustomJS(args=dict(source=source, slider=slider), code="""
    const data = source.data;
    const m = slider.value;
    const factor = data['Factor_Acumulado'];
    const cap = data['Capital_Evolucion'];

    for (let i = 0; i < factor.length; i++) {
        cap[i] = factor[i] * m;
    }
    source.change.emit();
""")

slider.js_on_change('value', callback)

show(column(slider, p))

# 5. Detector Avanzado de Anomalías Estructuradas (Plotly Graph Objects)

En este analisis se hizo un grafico de lineas para represntar el cambio diario de la acciones y se marcaron las anomalias con circulos color magenta, lo que al hacer hover nos dice el dia exacto en el que ocurrio esa anomalia, esto nos permite ver que paso ese dia y entender por que se genero ese outlier


In [ ]:
import plotly.graph_objects as go

anomalias = dfAluar_clean[(dfAluar_clean['Cambio_Diario'] > dfAluar_clean['Volatilidad_Alta'])]

fig_anomalias = go.Figure()

fig_anomalias.add_trace(go.Scatter(
    x=dfAluar_clean.index,
    y=dfAluar_clean['Cambio_Diario'],
    mode='lines',
    name='Rendimiento Diario',
    line=dict(color='#00ffff', width=1),
    opacity=0.5
))

fig_anomalias.add_trace(go.Scatter(
    x=anomalias.index,
    y=anomalias['Cambio_Diario'],
    mode='markers',
    name='🚨 Explosión de Volatilidad',
    marker=dict(
        color='#ff00ff',
        size=10,
        line=dict(color='white', width=1.5)
    )
))

fig_anomalias.update_layout(
    title="⚡ Detector Estadístico de Anomalías: Euforia en Aluar",
    template="plotly_dark",
    yaxis_tickformat='.2%',
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig_anomalias.show()

# Mapa de Calor Mensual 🗺️

Este análisis transforma la serie de tiempo en una matriz bidimensional (Año x Mes) para evaluar la estacionalidad de la acción. Utilizando un Heatmap divergente (donde el 0% es el punto de equilibrio), permite identificar visualmente patrones macroeconómicos históricos, revelando de forma instantánea qué trimestres suelen ser alcistas (verde) o bajistas (rojo) para la empresa, y dejando espacios vacíos automáticamente para los meses aún no cotizados.

**Basicamente Vamos a ver que meses del año son historicamente altos en cuanto acciones que(generan ganancias) o bajos que (generan perdidas)**

In [ ]:
import plotly.graph_objects as go

df_macro =  dfAluar_clean.copy()
#Seleccione el año y el mes de la primer columna 'index' y genere esas dos nuevas columnas
df_macro['Año'] = df_macro.index.year
df_macro['Mes'] = df_macro.index.month

#Agrupe los datos por año y mes, y calcule la media del su respectivo cambio diario
agrupandoAM = df_macro.groupby([df_macro['Año'],df_macro['Mes'] ])['Cambio_Diario'].mean()

#Converti los datos en un formato mas comprensible, unstack significa desarmar
mapa_datos = agrupandoAM.unstack()
# print(mapa_datos)


#Definimos los nombres de los meses para que el eje X quede más prolijo
meses_nombres = ['Ene',
                 'Feb', 'Mar', 'Abr', 'May',
                 'Jun', 'Jul', 'Ago', 'Sep',
                 'Oct', 'Nov', 'Dic']

# figura del Mapa de Calor
grafico = go.Figure(data=go.Heatmap(
    z=mapa_datos.values,
    x=meses_nombres,
    y=mapa_datos.index,
    colorscale='RdYlGn',
    zmid=0,
    text=mapa_datos.values,
    texttemplate="%{text:.2%}",
    hoverinfo="x+y+text"
))

#diseño oscuro
grafico.update_layout(
    title="🗺️ Mapa de Calor Mensual: Estacionalidad y Rendimiento de Aluar",
    template="plotly_dark",
    xaxis_title="Mes del Año",
    yaxis_title="Año Histórico",
    height=500,
    width=800
)

grafico.show()

